In [2]:
%load_ext autoreload
%autoreload 2

import analysis_tools.data_loading as dl
import analysis_tools.plotting as plt
import analysis_tools.signal_analysis as sa
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

In [43]:
# Load data
from pathlib import Path


dir_path = Path("/home/agilicious/catkin_ws/ros_logs/sim_body_rates")
command_path = dir_path / "cmd_body_rate_log.csv"
estimates_path = dir_path / "state_body_rate_log.csv"

command_df = pd.read_csv(command_path)
cmd_body_rate = command_df[["body_rate_x", "body_rate_y", "body_rate_z"]].to_numpy()
cmd_t = command_df["time"].to_numpy()
estimates_df = pd.read_csv(estimates_path)
state_body_rate = estimates_df[["body_rate_x", "body_rate_y", "body_rate_z"]].to_numpy()
state_t = estimates_df["time"].to_numpy()

# The csv stores multiple runs, so we need to split them at the t=0 mark.
zero_crossing = state_t == 0.0
which_section = 2
if np.any(zero_crossing):
    # Split the data at the zero crossing
    split_indices = np.where(zero_crossing)[0]
    if len(split_indices) > 1:
        print(split_indices[which_section], split_indices[which_section + 1])
        # If there are multiple runs, take the first one
        state_t = state_t[split_indices[which_section]:split_indices[which_section + 1]]
        state_body_rate = state_body_rate[split_indices[which_section]:split_indices[which_section + 1]]
        cmd_t = cmd_t[split_indices[which_section]:split_indices[which_section + 1]]
        cmd_body_rate = cmd_body_rate[split_indices[which_section]:split_indices[which_section + 1]]
    # else:
    #     # If there's only one run, use all data
    #     state_t = state_t[:split_indices[0]]
    #     state_body_rate = state_body_rate[:split_indices[0]]
    #     cmd_t = cmd_t[:split_indices[0]]
    #     cmd_body_rate = cmd_body_rate[:split_indices[0]]
    


In [36]:
# Plot pos Wx,y,z components
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Smoothed Vicon X", "Smoothed Vicon Y", "Smoothed Vicon Z"),
    vertical_spacing=0.1
)
plt.add_xyz_traces_stacked(fig, state_t, state_body_rate, label_prefix="State", line=dict(color='blue'))


fig.update_layout(
    title="Smoothed Vicon vs EKF Angular Velocity Components",
    xaxis_title="Time (s)",
    yaxis_title="Velocity (rad/s)",
    legend=dict(
        title="Legend",
        x=0.01,
        y=0.99,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='black',
        borderwidth=1
    )
)
fig.show()

In [24]:
# Plot pos Wx,y,z components
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Smoothed Vicon X", "Smoothed Vicon Y", "Smoothed Vicon Z"),
    vertical_spacing=0.1
)

plt.add_xyz_traces_stacked(fig, state_t, state_body_rate, label_prefix="Body Rate", line=dict(color='green'))
plt.add_xyz_traces_stacked(fig, cmd_t, cmd_body_rate, label_prefix="Body Rate Cmd", line=dict(color='blue'))
fig.show()


In [29]:
# --- Analyze delay between commands and state estimates body rates ---
# Get a specific time interval for analysis


state_body_rates = state_body_rate
t_states = state_t
cmd_body_rates = cmd_body_rate
t_cmds = cmd_t


i = 0
lag_samples, delay_x, cmd_x_aligned = sa.run_delay_analysis_nonuniform(state_body_rates[:,i], t_states, cmd_body_rates[:,i], t_cmds)
i = 1
lag_samples, delay_y, cmd_y_aligned = sa.run_delay_analysis_nonuniform(state_body_rates[:,i], t_states, cmd_body_rates[:,i], t_cmds)
i = 2
lag_samples, delay_z, cmd_z_aligned = sa.run_delay_analysis_nonuniform(state_body_rates[:,i], t_states, cmd_body_rates[:,i], t_cmds)

print(f"Delay X: {delay_x:.5f} s, Y: {delay_y:.5f} s, Z: {delay_z:.5f} s")
print(f"Avg Delay: {(delay_x + delay_y + delay_z) / 3:.5f} s")

cmd_body_rate_aligned = np.vstack((cmd_x_aligned, cmd_y_aligned, cmd_z_aligned)).T

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Aligned Body Rate", "Delay Analysis Correlation"),
    vertical_spacing=0.1
)
plt.add_xyz_traces_stacked(fig, t_cmds, cmd_body_rate_aligned, label_prefix="Commanded Body Rate", line=dict(color='blue'))
plt.add_xyz_traces_stacked(fig, t_states, state_body_rates, label_prefix="Aligned Body Rate", line=dict(color='green'))

fig.show()

Delay X: -0.08062 s, Y: -0.08232 s, Z: -0.18793 s
Avg Delay: -0.11696 s
